The below is a model that classifies SMS messages to Spam or Not Spam (ham). This dataset emerges from the combined efforts of Tiago A. Almeida and Akebo Yamakami at the School of Electrical and Computer Engineering at the University of Campinas in Brazil, and José María Gómez Hidalgo at the R&D Department of Optenet in Spain.

In [16]:
# Importing Libaries
import requests
import zipfile
import io
import os
import pandas as pd
import nltk
import re
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

In [3]:
# Downloading Dataset
url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
print("Downloading Dataset ...")
response = requests.get(url)
if response.status_code == 200:
    print("Downloaded successfully.")
else:
    print("Failed to download the dataset.")

# Extracting Dataset
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    print("Extracting Dataset ...")
    z.extractall("sms_spam_collection")

# Confirming Dataset Extraction
if os.listdir("sms_spam_collection"):
    print("Extracted successfully.")

Downloaded successfully.
Extracting Dataset ...
Extracted successfully.


# Preprocessing Data

In [4]:
# Load the dataset
df = pd.read_csv(
    "sms_spam_collection/SMSSpamCollection",
    sep="\t", # since dataset is stored in a tab-separated values (TSV) file format
    header=None,
    names=["label", "message"], # we are manually providing column names
)

In [5]:
# Quickly Inspecting Dataset
print("-------------------- HEAD --------------------")
print(df.head())
print("-------------------- DESCRIBE --------------------")
print(df.describe())
print("-------------------- INFO --------------------")
print(df.info())

-------------------- HEAD --------------------
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
-------------------- DESCRIBE --------------------
       label                 message
count   5572                    5572
unique     2                    5169
top      ham  Sorry, I'll call later
freq    4825                      30
-------------------- INFO --------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB
None


In [6]:
# Checking if there are any missing values.
print("Missing values:\n", df.isnull().sum())

Missing values:
 label      0
message    0
dtype: int64


We can see from the result above, that we don't have missing values.

In [7]:
# Check for duplicated entries
print("Duplicate entries:", df.duplicated().sum())

Duplicate entries: 403


We can see 403 duplicated entries. We need to drop duplicates.

In [8]:
# By default, df.drop_duplicates() uses keep='first', which means it preserves the first occurrence of a duplicated row and removes any subsequent copies.
df = df.drop_duplicates()

In [9]:
# Downloading NLTK data files
nltk.download("punkt_tab") # pre-trained data package in the Natural Language Toolkit
nltk.download("stopwords") # Stop words are frequently occurring words such as “is”, “the”, “and”, and “of” that may contribute little to certain text analysis tasks.

# Lowercasing the text ensures that the classifier treats words equally, regardless of their original casing.
df["message"] = df["message"].str.lower()
# Checking new lowercase data
print(df.head())

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


  label                                            message
0   ham  go until jurong point, crazy.. available only ...
1   ham                      ok lar... joking wif u oni...
2  spam  free entry in 2 a wkly comp to win fa cup fina...
3   ham  u dun say so early hor... u c already then say...
4   ham  nah i don't think he goes to usf, he lives aro...


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [10]:
# Remove non-essential punctuation and numbers to keep meaningful data, but symbols like $ and ! are generally added in spam messages so we keep them. We need to be balanced here.
df["message"] = df["message"].apply(lambda x: re.sub(r"[^a-z\s$!]", "", x))

In [11]:
# Tokenazation is a crucial step to do operations on words individually such as removing stop words and applying stemming
df["message"] = df["message"].apply(word_tokenize)

# Removing english language stop words
stop_words = set(stopwords.words("english"))
df["message"] = df["message"].apply(lambda x: [word for word in x if word not in stop_words])

# Stemming
stemmer = PorterStemmer()
df["message"] = df["message"].apply(lambda x: [stemmer.stem(word) for word in x])

# Tokens are useful for operations however, many machine-learning algorithms and vectorization techniques work best with raw text strings
# Rejoining tokens into a single string for feature extraction
df["message"] = df["message"].apply(lambda x: " ".join(x))

print(df.head())

  label                                            message
0   ham  go jurong point crazi avail bugi n great world...
1   ham                              ok lar joke wif u oni
2  spam  free entri wkli comp win fa cup final tkt st m...
3   ham                u dun say earli hor u c alreadi say
4   ham          nah dont think goe usf live around though


#Feature Extraction

In [12]:
# creating a matrix of term counts, where each row represents a message and each column corresponds to a term
# minimum documnet frequency = 1 so that the word must exist at least one to be included
# maximum document frequency removes words that are preset more that 90% of the time so that they don't provide limited differentiation
# ngram_range=(1, 2) capturing individual words and common word pairs to improve the ability to detect spam patterns
vectorizer = CountVectorizer(min_df=1, max_df=0.9, ngram_range=(1, 2))

# Fit and transform the message column
X = vectorizer.fit_transform(df["message"])
y = df["label"].apply(lambda x: 1 if x == "spam" else 0)  # Converting labels to 1 and 0

# Training and Evaluation

In [14]:
# The below pipeline ensures identical transformation and classification steps are chained and applied sequentially to our data
pipeline = Pipeline([
    ("vectorizer", vectorizer),
    ("classifier", MultinomialNB())
])

In [15]:
# Finding optimal parameter values for the classifier
# Define the parameter grid for hyperparameter tuning
param_grid = {
    "classifier__alpha": [0.01, 0.1, 0.15, 0.2, 0.25, 0.5, 0.75, 1.0]
}

# Perform the grid search with 5-fold cross-validation and the F1-score as metric
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="f1"
)

# Fit the grid search on the full dataset
grid_search.fit(df["message"], y)

# Extract the best model identified by the grid search
best_model = grid_search.best_estimator_
print("Best model parameters:", grid_search.best_params_)

Best model parameters: {'classifier__alpha': 0.25}


#Evaluation

In [17]:
# Example SMS messages for evaluation
new_messages = [
    "Congratulations! You've won a $1000 Walmart gift card. Go to http://bit.ly/1234 to claim now.",
    "Hey, are we still meeting up for lunch today?",
    "Urgent! Your account has been compromised. Verify your details here: www.fakebank.com/verify",
    "Reminder: Your appointment is scheduled for tomorrow at 10am.",
    "FREE entry in a weekly competition to win an iPad. Just text WIN to 80085 now!",
]

In [18]:
# We must preprocess the new messages using the same steps applied during training.
# Since we explained each step in the Preprocessing section, we can now define the steps as a function so we can reuse it.
# Preprocess function that mirrors the training-time preprocessing
def preprocess_message(message):
    message = message.lower()
    message = re.sub(r"[^a-z\s$!]", "", message)
    tokens = word_tokenize(message)
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [stemmer.stem(word) for word in tokens]
    return " ".join(tokens)

# Preprocess and vectorize the new_messages
processed_messages = [preprocess_message(msg) for msg in new_messages]
X_new = best_model.named_steps["vectorizer"].transform(processed_messages)

# Make predictions
predictions = best_model.named_steps["classifier"].predict(X_new)
prediction_probabilities = best_model.named_steps["classifier"].predict_proba(X_new) # this will show us the models confidence in its prediction

In [19]:
# Display predictions and probabilities for each evaluated message
for i, msg in enumerate(new_messages):
    prediction = "Spam" if predictions[i] == 1 else "Not-Spam"
    spam_probability = prediction_probabilities[i][1]  # Probability of being spam
    ham_probability = prediction_probabilities[i][0]   # Probability of being not spam

    print(f"Message: {msg}")
    print(f"Prediction: {prediction}")
    print(f"Spam Probability: {spam_probability:.2f}")
    print(f"Not-Spam Probability: {ham_probability:.2f}")
    print("-" * 50)

Message: Congratulations! You've won a $1000 Walmart gift card. Go to http://bit.ly/1234 to claim now.
Prediction: Spam
Spam Probability: 1.00
Not-Spam Probability: 0.00
--------------------------------------------------
Message: Hey, are we still meeting up for lunch today?
Prediction: Not-Spam
Spam Probability: 0.00
Not-Spam Probability: 1.00
--------------------------------------------------
Message: Urgent! Your account has been compromised. Verify your details here: www.fakebank.com/verify
Prediction: Spam
Spam Probability: 0.96
Not-Spam Probability: 0.04
--------------------------------------------------
Message: Reminder: Your appointment is scheduled for tomorrow at 10am.
Prediction: Not-Spam
Spam Probability: 0.00
Not-Spam Probability: 1.00
--------------------------------------------------
Message: FREE entry in a weekly competition to win an iPad. Just text WIN to 80085 now!
Prediction: Spam
Spam Probability: 1.00
Not-Spam Probability: 0.00
----------------------------------